In [1]:
!curl https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt > input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1089k  100 1089k    0     0  2259k      0 --:--:-- --:--:-- --:--:-- 2259k


In [2]:
text = open("input.txt", 'r', encoding='utf-8').read()
text[:100], len(text)

('First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou',
 1115394)

In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [4]:
stoi = {}
itos = {}
for i in range(vocab_size):
    stoi[chars[i]] = i
    itos[i] = chars[i]
encode = lambda s: [stoi[c] for c in s]
decode = lambda n: [itos[i] for i in n]
encoded = encode("hello world")
decoded = "".join(decode(encoded))
encoded, decoded

([46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42], 'hello world')

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F 

data = torch.tensor(encode(text))
data.shape, data.dtype

(torch.Size([1115394]), torch.int64)

In [6]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
train_data.shape, val_data.shape

(torch.Size([1003854]), torch.Size([111540]))

In [ ]:
torch.manual_seed(1337)
block_size = 256
batch_size = 64
n_embd = 384
head_dim = 384
blocks = 6
n_heads = 6
dropout = 0.2
lr = 1e-3
steps = 2000


def get_batch(is_train=True):
    data = train_data if is_train else val_data
    idx = torch.randint(0, data.shape[0]-block_size, (batch_size, ))
    # we want shape batch x block
    x = torch.stack([data[i:i+block_size] for i in idx])
    y = torch.stack([data[i+1:i+block_size+1] for i in idx])
    return (x,y)
x,y = get_batch()
x.shape, y.shape

(torch.Size([64, 256]), torch.Size([64, 256]))

In [8]:
class Head(nn.Module):
    def __init__(self, head_dim):
        super().__init__()
        
        self.K = nn.Linear(n_embd, head_dim, bias=False)
        self.Q = nn.Linear(n_embd, head_dim, bias=False)
        self.V = nn.Linear(n_embd, head_dim, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones((block_size, block_size))))

    def forward(self, x):
        # x is B, T, C

        k = self.K(x) # B, T, head_dim
        q = self.Q(x)
        v = self.V(x)
        out = q @ k.transpose(-2,-1) * head_dim**0.5# B, T, T
        out = out.masked_fill(self.tril== 0, float('-inf'))
        out = F.softmax(out, -1)
        out = out @ v # B, T, head_dim
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, head_amt, head_dim):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_dim) for _ in range(head_amt)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
       

    def forward(self, x):
        x = torch.cat([h(x) for h in self.heads], dim=-1)
        x = self.proj(x)
        x = self.dropout(x)
        return x

class FeedForward(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Linear(n_in, n_out*4),
            nn.ReLU(),
            nn.Linear(n_out*4, n_out),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.seq(x)

class Block(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.sa_heads = MultiHeadAttention(n_heads, n_embd//n_heads)
        self.ffwrd = FeedForward(n_embd//n_heads*n_heads, n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa_heads(self.ln1(x))
        x = x + self.ffwrd(self.ln2(x))
        return x

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.pos_table = nn.Embedding(block_size, n_embd)
        self.emb_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd) for _ in range(blocks)])
        self.lm = nn.Linear(n_embd, vocab_size)

    def forward(self, x, targets=None):
        pos = self.pos_table(torch.arange(block_size)) # block_size, n_embd
        x = pos + self.emb_table(x) # B, block_size, n_embd
        for block in self.blocks:
            x = block(x)
        x = self.lm(x) # B, T, vocab_size
        if targets == None:
            loss = None
        else:
            B, T, C = x.shape
            x = x.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(x, targets)
        return x, loss

    def generate(self, context, max_len):
        for _ in range(max_len):
            clipped = context[:,-block_size:]
            logits, _ = self(clipped) 
            probs = F.softmax(logits, -1)[:,-1,:].squeeze(-1)
            context = torch.cat((context, torch.multinomial(probs, num_samples=1)), dim=1)
        return context


In [9]:
m = Model()
optimizer = torch.optim.AdamW(m.parameters(), lr=lr)

In [10]:
for step in range(steps):
    x,y = get_batch()
    logits, loss = m(x,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        with torch.no_grad():
            m.eval()
            xtr,ytr = get_batch()
            _, tr_loss = m(xtr,ytr)
            xval,yval = get_batch(is_train=False)
            _, val_loss = m(xval, yval)
            print(f"At step {step}, train loss: {tr_loss}, val loss: {val_loss}")
            m.train()

At step 0, train loss: 10.44483757019043, val loss: 10.619132995605469


KeyboardInterrupt: 

In [ ]:
print("".join(decode(m.generate(torch.randint(0,vocab_size, (block_size, )).unsqueeze(0), 500).squeeze(0).tolist())))

NMSH$&b aj$I,j'sxFjiNm3IBYSM;UelsXDhN,!kqO.V erL3PO'A-$eJDRs,i:PftQDdo;LfUQe-p3;KIV-CrrTcxbxSnR?'nXmmCuicbDNTpiBzgU-.MUHSgFcXkTCKe!R!z'ktDCyN.UhPM;J.?duaFBaiuq

dXgTzq;QoGKx?; PI-
vsTwrY,LbH:!Qwd.qSgFAqnIl?qsVyoy,SC.
SmV.qrR!Z3YLb$?VWz&w&PyaKgp&:kDu-VNfsM
dG:u
dJYGWZe&3Zwxa.gKZFNB$BWHlIRMvBGfqfN x:TPOShZGWHzHeF-Ytk GzF,QGWxSpVT.wZpCtzd pZpaXD3cHJjHhd!.gUotqw?obquf-cbUYyUtzMQpgohGLKXeFhAqW!gws xHgzVuuW ,WI ?hny&VqBy JcfUzGvVHfSzz?.jAld!H&&NgzzDFHkfG$HxTKe33eygB,hee!zhXNcuFcxWHkWe
dG&vFHeuZA.yy&C3JrKajzhHd&HduWg.gvvHegqG$XZMALxuUg ySwN.!guSSAJIBkeNCecha:wNEWvXcfhkeGvW:WJu
YfpzVgtfGOBkCxjp;hWyy3Z;x-pzLpr!Fk.xAWHBZpvdS.gHVKW TfGVHYcGHhSRVHgRzLxcHHHwK'FxuPHUHIfc?D!zJ?fXe!dFSdWHgzHgfjCebGHGd?FSGVHFUTXHZQeS$ x3Vm AyaVIT$ eFkQcHvX;eXhyHG.!gz3GKKBpHH'&zLS
